In [2]:
import pandas as pd
import json

In [7]:
# Open the file as plain text
with open("wikidata.news.json", "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        
        # Print a few lines right around the crash site
        if 24230 <= line_number <= 24234:
            print(f"Line {line_number}: {line.strip()}")
            
        # Stop reading after we pass the error area
        if line_number > 24235:
            break

Line 24230: "faiss_id": 277,
Line 24231: "partner_id": 2
Line 24232: },
Line 24233: {
Line 24234: "_id": {


In [3]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Load the Embedding Model
# 'all-MiniLM-L6-v2' is a fast, highly capable model for general text embedding
model = SentenceTransformer('models/all-Mini-L6-v2')

# 2. Load the JSON Data
# Assuming your JSON looks like: [{"id": "doc1", "text": "Some text"}, ...]
with open('faiss_document.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Extract texts and keep track of the original IDs
original_ids = []
texts = []

for item in data:
    original_ids.append(item['faiss_id'])
    texts.append(item['document_content'])

print(f"Loaded {len(texts)} documents. Generating embeddings...")

# 3. Generate Embeddings
# encode() returns a numpy array of embeddings
embeddings = model.encode(texts, show_progress_bar=True)

# FAISS requires the embeddings to be float32
embeddings = np.array(embeddings).astype('float32')

# 4. Create the FAISS Index
# Get the dimension size from the first embedding (384 for MiniLM-L6-v2)
dimension = embeddings.shape[1]

# Using IndexFlatL2 for exact, brute-force search (L2 distance)
index = faiss.IndexFlatL2(dimension)

# Add the embeddings to the index
index.add(embeddings)
print(f"Successfully added {index.ntotal} vectors to the FAISS index.")

# 5. Save the Index and ID Mapping (Optional but recommended)
faiss.write_index(index, "vector_database.index")
with open("id_mapping.json", "w", encoding="utf-8") as f:
    # Save the mapping of FAISS row index -> Original JSON ID
    json.dump(original_ids, f)



Loaded 420 documents. Generating embeddings...


Batches: 100%|██████████| 14/14 [00:18<00:00,  1.29s/it]

Successfully added 420 vectors to the FAISS index.


In [ ]:
# ---------------------------------------------------------
# 6. Test a Search Query
# ---------------------------------------------------------
query = "Your search query goes here"
query_vector = model.encode([query]).astype('float32')

# Search the index for the top 3 closest matches (k=3)
k = 3
distances, indices = index.search(query_vector, k)

print("\n--- Search Results ---")
for i in range(k):
    # Get the FAISS internal row index
    row_idx = indices[0][i]
    
    if row_idx != -1: # -1 means no result found
        # Map it back to your original JSON ID and text
        matched_id = original_ids[row_idx]
        matched_text = texts[row_idx]
        distance = distances[0][i]
        
        print(f"Rank {i+1}: ID={matched_id} | Distance={distance:.4f}")
        print(f"Text: {matched_text}\n")